<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 05 — Regressão Logística
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — Probabilidades e o Segundo Modelo no Titanic
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>
<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📈 2º Modelo do Curso</span>
</div>


## Onde estamos?

Na Aula 04 treinamos a Regressão Linear — nosso primeiro modelo, usado para prever
**um número** (a tarifa da passagem). Hoje apresentamos o **segundo modelo do curso,
e o primeiro de classificação: a Regressão Logística**.

Apesar do nome "regressão", ela é um **algoritmo de classificação**. Vamos usá-la
para prever se um passageiro **sobreviveu ou não** — mas em vez de só responder
sim/não, ela nos dá uma probabilidade:

> *Em vez de só dizer "sobreviveu ou não", a Regressão Logística diz:*
> *"Esse passageiro tem **74% de chance** de ter sobrevivido."*

### Roteiro de hoje

| Parte | Tema |
|-------|------|
| **1** | Por que não usar Regressão Linear para classificar? |
| **2** | A função sigmoide — transformando números em probabilidade |
| **3** | Nosso segundo modelo — `fit`, `predict`, `predict_proba` e coeficientes |
| **4** | O limiar de decisão |
| **5** | Regressão Linear vs Regressão Logística — comparação final |

> **Tempo estimado: 35 minutos**

<div style="background:#d1ecf1; border-left:5px solid #0c5460; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#0c5460;">Dica de uso profissional:</strong>
<span style="color:#0c5460;"> a Regressão Logística é frequentemente usada como <strong>baseline</strong> em projetos reais de classificação. Se um modelo mais complexo não a superar por uma margem relevante, geralmente não vale a complexidade extra.</span>
</div>


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize":  (10, 5),
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
})
sns.set_theme(style="whitegrid", palette="muted")

from sklearn.model_selection import train_test_split

# ── Carregando e preparando o Titanic (mesma limpeza das aulas anteriores) ────
df = sns.load_dataset("titanic").copy()

df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

df["tamanho_familia"] = df["sibsp"] + df["parch"] + 1
df["sex_enc"]         = (df["sex"] == "female").astype(int)

# 5 features simples e fáceis de explicar
FEATURES = ["pclass", "sex_enc", "age", "tamanho_familia", "fare"]

X = df[FEATURES].copy()
y = df["survived"].copy()

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("✅ Dataset pronto!")
print(f"   Treino: {len(X_treino)} passageiros  |  Teste: {len(X_teste)} passageiros")
print(f"   Features usadas: {FEATURES}")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Por Que Não Usar Regressão Linear para Classificar?</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"O problema não é a matemática — é o que a saída representa."</p>
    </div>
</div>

### O problema concreto

A Regressão **Linear** produz uma reta que prevê um número contínuo:
```
ŷ = β₀ + β₁·x₁ + β₂·x₂ + ...    → ŷ pode ser qualquer número: -∞ a +∞
```

Se tentarmos usá-la para classificação (prever 0 ou 1), dois problemas aparecem:

**Problema 1 — A saída sai de [0, 1]**
O modelo pode prever 1.8 ou -0.3. O que significa "probabilidade de 180%"?

**Problema 2 — A reta força linearidade onde não existe**
A chance de sobreviver não cresce indefinidamente com a tarifa — há um teto natural
em 100% e um piso em 0%.

**A solução: a função sigmoide.** A Regressão Logística aplica uma transformação
na saída linear que garante o resultado sempre entre 0 e 1. O gráfico abaixo mostra
lado a lado o problema e a solução.


In [ ]:
# Demonstrando o problema da regressão linear para classificação
from sklearn.linear_model import LinearRegression
from scipy.special import expit as sigmoid

np.random.seed(42)

# Simulando: tarifa paga vs sobrevivência (binário)
n = 80
tarifas_nao_sobrev = np.random.normal(20, 15, n//2).clip(0, 100)
tarifas_sobrev     = np.random.normal(60, 25, n//2).clip(0, 200)
tarifas = np.concatenate([tarifas_nao_sobrev, tarifas_sobrev]).reshape(-1, 1)
labels  = np.array([0]*(n//2) + [1]*(n//2))

# Ajustando regressão LINEAR
lr_linear = LinearRegression()
lr_linear.fit(tarifas, labels)

x_plot = np.linspace(-10, 210, 300).reshape(-1, 1)
y_linear = lr_linear.predict(x_plot)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Por que Regressão Linear Falha na Classificação", fontweight="bold")

# Gráfico 1: problema da regressão linear
ax1.scatter(tarifas_nao_sobrev, np.zeros(n//2), color="#e94560",
            alpha=0.5, s=40, label="Não Sobreviveu (0)")
ax1.scatter(tarifas_sobrev, np.ones(n//2), color="#0f3460",
            alpha=0.5, s=40, label="Sobreviveu (1)")
ax1.plot(x_plot, y_linear, color="#f0a500", linewidth=2.5, label="Regressão Linear")
ax1.axhline(0, color="gray", linestyle="--", alpha=0.4)
ax1.axhline(1, color="gray", linestyle="--", alpha=0.4)
ax1.axhspan(-0.5, 0, alpha=0.05, color="#e94560", label="Prob. negativa ❌")
ax1.axhspan(1, 1.5, alpha=0.05, color="#e94560")
ax1.set_xlabel("Tarifa (£)")
ax1.set_ylabel("Sobreviveu (0 ou 1)")
ax1.set_title("Regressão LINEAR\nPode prever valores fora de [0, 1] ❌")
ax1.set_ylim(-0.5, 1.5)
ax1.legend(fontsize=8)

# Gráfico 2: o que queremos — probabilidade entre 0 e 1
beta0, beta1 = -3.0, 0.05
x_vals = np.linspace(-10, 210, 300)
y_logit = sigmoid(beta0 + beta1 * x_vals)

ax2.scatter(tarifas_nao_sobrev, np.zeros(n//2), color="#e94560",
            alpha=0.5, s=40, label="Não Sobreviveu (0)")
ax2.scatter(tarifas_sobrev, np.ones(n//2), color="#0f3460",
            alpha=0.5, s=40, label="Sobreviveu (1)")
ax2.plot(x_vals, y_logit, color="#0f3460", linewidth=2.5,
         label="Regressão LOGÍSTICA")
ax2.axhline(0.5, color="#f0a500", linestyle="--", alpha=0.7, label="Limiar 0.5")
ax2.set_xlabel("Tarifa (£)")
ax2.set_ylabel("P(Sobreviveu = 1)")
ax2.set_title("Regressão LOGÍSTICA\nSaída sempre entre 0 e 1 ✅")
ax2.set_ylim(-0.1, 1.1)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Observe:")
print("  Linear: pode prever -0.2 ou 1.4 — sem sentido como probabilidade")
print("  Logística: sempre entre 0 e 1 — interpretável como probabilidade")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">A Função Sigmoide — O Coração da Regressão Logística</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Uma função elegante que transforma qualquer número em uma probabilidade."</p>
    </div>
</div>

### A equação em duas partes

```
         1
P(Y=1) = ─────────────────────────────────
         1 + e^−z       onde  z = β₀ + β₁·X₁ + β₂·X₂ + ...
```

**Parte 1 — a combinação linear** (igual à regressão linear): `z = β₀ + β₁·X₁ + ...`
— pode ser qualquer número.

**Parte 2 — a sigmoide** transforma esse `z` em probabilidade — sempre entre 0 e 1.

| z muito negativo | z = 0 | z muito positivo |
|------------------|-------|-----------------|
| P ≈ 0 (não sobreviveu) | P = 0,5 (50/50) | P ≈ 1 (sobreviveu) |


In [ ]:
# Visualizando a função sigmoide
z = np.linspace(-8, 8, 400)
p = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8, 5))
plt.plot(z, p, color="#0f3460", linewidth=3)
plt.axhline(0.5, color="#e94560", linestyle="--", linewidth=1.5, label="P = 0.5 (limiar padrão)")
plt.axvline(0.0, color="#f0a500", linestyle="--", linewidth=1.5, label="z = 0 → P = 0.5")
plt.xlabel("z  (combinação linear das features)")
plt.ylabel("P(Y=1)  — probabilidade")
plt.title("σ(z) = 1 / (1 + e^−z)", fontweight="bold")
plt.legend()
plt.ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()


### De onde vem o "z" na prática?

O `z` não é um número aleatório — ele vem da combinação das features de um
passageiro real, ponderadas pelos coeficientes que o modelo aprende. Vamos ver
isso concretamente, com um `z` calculado à mão para um passageiro fictício,
**antes** de treinar qualquer modelo de verdade.


In [ ]:
# De uma passageira real para o valor de z (com coeficientes "de brincadeira",
# só para ilustrar o mecanismo — o modelo real vai aprender os seus na Parte 3)
passageira = {"pclass": 1, "sex_enc": 1, "age": 28, "tamanho_familia": 1, "fare": 80}

# Coeficientes ilustrativos (positivos ajudam a sobreviver, negativos atrapalham)
beta0           = -1.0
beta_pclass     = -0.8   # classe mais alta numericamente (3ª) reduz a chance
beta_sex        =  2.5   # ser mulher aumenta muito a chance
beta_age        = -0.02
beta_familia    =  0.1
beta_fare       =  0.01

z = (beta0
     + beta_pclass  * passageira["pclass"]
     + beta_sex     * passageira["sex_enc"]
     + beta_age     * passageira["age"]
     + beta_familia * passageira["tamanho_familia"]
     + beta_fare    * passageira["fare"])

p_sobrevive = 1 / (1 + np.exp(-z))

print(f"Passageira: {passageira}")
print(f"\nz = {beta0} + ({beta_pclass})×{passageira['pclass']} + ({beta_sex})×{passageira['sex_enc']} "
      f"+ ({beta_age})×{passageira['age']} + ({beta_familia})×{passageira['tamanho_familia']} "
      f"+ ({beta_fare})×{passageira['fare']}")
print(f"z = {z:.3f}")
print(f"\nP(Sobreviveu) = σ({z:.3f}) = {p_sobrevive:.1%}")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — Calcule manualmente P(Y=1) para os valores de z abaixo usando a fórmula σ(z) = 1 / (1 + e^−z). O que cada resultado significa no contexto do Titanic?</span></div>

In [ ]:
# ✏️ Calcule P(Y=1) para cada valor de z
valores_z = [-5, -1, 0, 1, 5]

print("Cálculo manual da sigmoide:")
print(f"{'z':>6}  {'P(Y=1) calculado':>18}  {'P(Y=1) numpy':>14}  Interpretação")
print("-" * 70)

for z_val in valores_z:
    # ✏️ p_manual = ???  (use a fórmula: 1 / (1 + e^-z))
    p_numpy = 1 / (1 + np.exp(-z_val))

    if p_numpy < 0.3:
        interpretacao = "Alta chance de NAO sobreviver"
    elif p_numpy > 0.7:
        interpretacao = "Alta chance de sobreviver"
    else:
        interpretacao = "Incerto (zona intermediaria)"

    print(f"{z_val:>6}  {'???':>18}  {p_numpy:>14.4f}  {interpretacao}")


In [ ]:
# ── GABARITO DA MISSÃO 1 (descomente para ver) ───────────────────────────────
# import math
# print("Gabarito — Cálculo Manual da Sigmoide:")
# for z_val in [-5, -1, 0, 1, 5]:
#     e_menos_z = math.exp(-z_val)
#     p = 1 / (1 + e_menos_z)
#     print(f"z={z_val:>3}:  1/(1+e^{-z_val}) = 1/(1+{e_menos_z:.3f}) = {p:.4f}")
# print()
# print("Interpretação no Titanic:")
# print("  z=-5 → P=0.7%  → quase certeza de NÃO sobreviver")
# print("  z=-1 → P=26.9% → mais provavelmente NÃO sobreviveu")
# print("  z= 0 → P=50.0% → exatamente na fronteira de decisão")
# print("  z=+1 → P=73.1% → mais provavelmente sobreviveu")
# print("  z=+5 → P=99.3% → quase certeza de ter sobrevivido")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Nosso Segundo Modelo — fit, predict e predict_proba</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"A grande novidade: agora temos probabilidades, não só classes."</p>
    </div>
</div>

Chega de coeficientes inventados — vamos deixar o `scikit-learn` encontrar os
coeficientes reais a partir dos dados do Titanic. A interface é a mesma que
você já usou na Aula 04: `fit()`, `predict()`. A novidade é o `predict_proba()`,
que retorna a probabilidade direto da sigmoide (a Regressão Linear não tem
esse método, porque ela não produz probabilidades).


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_treino_sc = scaler.fit_transform(X_treino)
X_teste_sc  = scaler.transform(X_teste)

modelo_lr = LogisticRegression(max_iter=1000, random_state=42)
modelo_lr.fit(X_treino_sc, y_treino)

print("✅ Regressão Logística treinada!")
print(f"   Intercepto (β₀): {modelo_lr.intercept_[0]:.4f}")
print(f"   Acurácia no teste: {modelo_lr.score(X_teste_sc, y_teste):.1%}")


In [ ]:
# predict vs predict_proba — a grande diferença
y_classes = modelo_lr.predict(X_teste_sc)          # classe: 0 ou 1
y_probas  = modelo_lr.predict_proba(X_teste_sc)    # [P(classe=0), P(classe=1)]

print("Comparando predict() vs predict_proba() — primeiros 10 passageiros")
print(f"  {'#':>3}  {'Real':>10}  {'Previsto':>10}  {'P(Sobreviveu)':>14}  {'Acerto?'}")
print("  " + "-"*55)
for i in range(10):
    real  = y_teste.values[i]
    prev  = y_classes[i]
    p_sim = y_probas[i][1]
    ok    = "✅" if real == prev else "❌"
    r_txt = "Sobreviveu" if real == 1 else "Não Sobrev."
    p_txt = "Sobreviveu" if prev == 1 else "Não Sobrev."
    print(f"  {i+1:>3}  {r_txt:>10}  {p_txt:>10}  {p_sim:>14.1%}  {ok}")

print()
print("💡 Quando P(Sobreviveu) > 50%, o modelo prevê 'Sobreviveu'.")
print("   Esse limiar de 50% pode ser ajustado — veremos isso na Parte 4!")


### Interpretando os coeficientes

Assim como na Regressão Linear, cada feature tem um coeficiente β. A diferença
é que aqui a leitura é sobre **probabilidade**:

```
Coeficiente POSITIVO → quando essa feature aumenta, P(sobreviveu) AUMENTA
Coeficiente NEGATIVO → quando essa feature aumenta, P(sobreviveu) DIMINUI
```

Como as features estão normalizadas, os coeficientes são comparáveis entre si.


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — Antes de executar a célula abaixo, escreva sua hipótese: qual feature você acha que terá o maior coeficiente POSITIVO? E o maior NEGATIVO? Depois compare com o resultado.</span></div>

*✏️ Minha hipótese — maior coeficiente POSITIVO: `???`*

*✏️ Minha hipótese — maior coeficiente NEGATIVO: `???`*


In [ ]:
# Extraindo e visualizando os coeficientes
coeficientes = pd.DataFrame({
    "feature":     FEATURES,
    "coeficiente": modelo_lr.coef_[0]
}).sort_values("coeficiente", ascending=False)

plt.figure(figsize=(8, 5))
cores = ["#0f3460" if v > 0 else "#e94560" for v in coeficientes["coeficiente"]]
plt.barh(coeficientes["feature"], coeficientes["coeficiente"], color=cores, edgecolor="white")
plt.axvline(0, color="black", linewidth=1.2)
plt.xlabel("Coeficiente β (normalizado)")
plt.title("Coeficientes da Regressão Logística\n(Azul = aumenta chance de sobreviver | Vermelho = diminui)",
          fontweight="bold")
plt.tight_layout()
plt.show()

print(coeficientes.to_string(index=False))


In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver a interpretação completa) ───────
# print("Gabarito — Interpretação dos coeficientes:")
# print("  sex_enc costuma ter o maior coeficiente POSITIVO:")
# print("  mulheres tiveram taxa de sobrevivência muito maior (protocolo")
# print("  'mulheres e crianças primeiro').")
# print()
# print("  pclass costuma ter um coeficiente NEGATIVO forte:")
# print("  quanto maior o número da classe (3ª > 2ª > 1ª), menor a chance")
# print("  de sobrevivência.")
# print()
# print("O modelo não foi programado com essas regras — ele as aprendeu dos dados.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">O Limiar de Decisão — Quando 0,5 Não é a Melhor Escolha</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"A probabilidade é do modelo. A decisão é sua."</p>
    </div>
</div>

O modelo gera probabilidades. Para virar uma classe (Sim/Não), usamos uma regra:

```
Se P(Sobreviveu) > LIMIAR  →  prevê "Sobreviveu"
Se P(Sobreviveu) ≤ LIMIAR  →  prevê "Não Sobreviveu"
```

O padrão é `LIMIAR = 0.5`, mas isso é uma **escolha**, não uma regra fixa:

| Situação | Ajuste | Efeito |
|----------|--------|--------|
| Falso Negativo é muito caro (ex: diagnóstico médico) | **Baixar** o limiar | Mais alertas → maior Recall, menor Precisão |
| Falso Positivo é muito caro (ex: alarme falso caro) | **Subir** o limiar | Menos alarmes → maior Precisão, menor Recall |


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Testando diferentes limiares
limiares    = np.arange(0.1, 0.91, 0.05)
precisoes   = []
recalls     = []
f1s         = []

prob_sobrev = modelo_lr.predict_proba(X_teste_sc)[:, 1]

for limiar in limiares:
    y_pred_limiar = (prob_sobrev >= limiar).astype(int)
    precisoes.append(precision_score(y_teste, y_pred_limiar, zero_division=0))
    recalls.append(recall_score(y_teste, y_pred_limiar, zero_division=0))
    f1s.append(f1_score(y_teste, y_pred_limiar, zero_division=0))

melhor_limiar_f1 = limiares[np.argmax(f1s)]

plt.figure(figsize=(9, 5))
plt.plot(limiares, precisoes, "o-", color="#0f3460", label="Precisão")
plt.plot(limiares, recalls,   "s-", color="#e94560", label="Recall")
plt.plot(limiares, f1s,       "^-", color="#f0a500", label="F1-Score")
plt.axvline(0.5, color="gray", linestyle="--", label="Limiar padrão (0.5)")
plt.axvline(melhor_limiar_f1, color="#2ecc71", linestyle="--",
            label=f"Melhor F1 ({melhor_limiar_f1:.2f})")
plt.xlabel("Limiar de Decisão"); plt.ylabel("Valor da Métrica")
plt.title("Efeito do Limiar nas Métricas", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Limiar que maximiza F1-Score: {melhor_limiar_f1:.2f}  (F1={max(f1s):.4f})")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — Usando <code>(prob_sobrev >= melhor_limiar_f1).astype(int)</code>, compare com o limiar padrão de 0.5: (a) o F1 melhora? (b) o Recall aumentou ou diminuiu? (c) a Precisão aumentou ou diminuiu? Explique o trade-off que você observou.</span></div>

In [ ]:
# ✏️ MISSÃO 3 — complete a análise
# y_pred_melhor = (prob_sobrev >= melhor_limiar_f1).astype(int)
# y_pred_padrao = modelo_lr.predict(X_teste_sc)

# ✏️ Compare f1_score, recall_score e precision_score dos dois
# f1_padrao  = ???
# f1_melhor  = ???
# print(f"Limiar 0.50:  F1={f1_padrao:.4f}")
# print(f"Melhor limiar: F1={f1_melhor:.4f}")


In [ ]:
# ── GABARITO DA MISSÃO 3 (descomente para ver) ───────────────────────────────
# from sklearn.metrics import accuracy_score
# y_pred_050    = modelo_lr.predict(X_teste_sc)
# y_pred_melhor = (prob_sobrev >= melhor_limiar_f1).astype(int)
#
# print(f"{'Métrica':<12} {'Limiar 0.50':>12} {'Limiar '+str(round(melhor_limiar_f1,2)):>14}")
# for nome, func in [("Acurácia", accuracy_score), ("Precisão", precision_score),
#                     ("Recall", recall_score), ("F1-Score", f1_score)]:
#     v1 = func(y_teste, y_pred_050)
#     v2 = func(y_teste, y_pred_melhor)
#     print(f"{nome:<12} {v1:>12.4f} {v2:>14.4f}  ({v2-v1:+.4f})")
#
# print()
# print("Trade-off: baixar o limiar aumenta o Recall (encontra mais sobreviventes")
# print("reais) mas diminui a Precisão (erra prevendo sobrevivência demais).")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 5</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Regressão Linear vs Regressão Logística — Comparação Final</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Não existe modelo melhor em geral — existe modelo mais adequado para cada problema."</p>
    </div>
</div>

Na Parte 1 você viu, de forma visual, por que a Regressão Linear é inadequada
para classificação. Agora vamos comparar os dois modelos **com números reais**,
usando a Regressão Linear como um "classificador improvisado" (arredondando a
previsão contínua para 0 ou 1).

Uma métrica nova aparece aqui: a **AUC-ROC**.

```
AUC = área sob a curva que compara Recall vs Falso Positivo em todos os limiares
  AUC = 0.5 → modelo aleatório (inútil)
  AUC = 1.0 → modelo perfeito
```


In [ ]:
# Treinando a Regressão Linear como "classificador improvisado" para comparação
reg_linear_ref = LinearRegression()
reg_linear_ref.fit(X_treino_sc, y_treino)

pred_continua_rl = reg_linear_ref.predict(X_teste_sc)
y_pred_rl = np.clip(np.round(pred_continua_rl), 0, 1).astype(int)
y_pred_lr = modelo_lr.predict(X_teste_sc)

# "Probabilidade" da Regressão Linear = o próprio valor previsto, limitado a [0,1]
# (não é uma probabilidade de verdade — só serve para desenhar a curva ROC)
y_prob_rl = np.clip(pred_continua_rl, 0, 1)
y_prob_lr = modelo_lr.predict_proba(X_teste_sc)[:, 1]

print("✅ Regressão Linear (adaptada) pronta para comparação.")


In [ ]:
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)

print("COMPARAÇÃO — Regressão Linear (adaptada) vs Regressão Logística")
print(f"  {'Métrica':<12} {'Reg. Linear':>12} {'Reg. Logística':>16}")
print("  " + "-"*44)
for nome, func in [("Acurácia", accuracy_score), ("Precisão", precision_score),
                    ("Recall", recall_score), ("F1-Score", f1_score)]:
    v_rl = func(y_teste, y_pred_rl)
    v_lr = func(y_teste, y_pred_lr)
    print(f"  {nome:<12} {v_rl:>12.4f} {v_lr:>16.4f}")

auc_rl = roc_auc_score(y_teste, y_prob_rl)
auc_lr = roc_auc_score(y_teste, y_prob_lr)
print(f"  {'AUC-ROC':<12} {auc_rl:>12.4f} {auc_lr:>16.4f}")


In [ ]:
# Curvas ROC sobrepostas
plt.figure(figsize=(7, 6))
for y_prob, nome, cor in [(y_prob_rl, "Reg. Linear (adaptada)", "#0f3460"),
                           (y_prob_lr, "Reg. Logística", "#e94560")]:
    fpr, tpr, _ = roc_curve(y_teste, y_prob)
    auc = roc_auc_score(y_teste, y_prob)
    plt.plot(fpr, tpr, color=cor, linewidth=2.2, label=f"{nome} (AUC={auc:.3f})")

plt.plot([0,1], [0,1], "k--", linewidth=1, alpha=0.5, label="Aleatório (AUC=0.5)")
plt.xlabel("Taxa de Falso Positivo")
plt.ylabel("Taxa de Verdadeiro Positivo (Recall)")
plt.title("Curva ROC — quanto mais alto e à esquerda, melhor", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 (final) — Olhando a tabela de métricas e a curva ROC: (a) qual modelo você escolheria para este problema e por quê? (b) por que a Regressão Linear, mesmo adaptada, tende a ser pior classificadora do que a Regressão Logística?</span></div>

*✏️ (a) Eu escolheria: `???` porque: `???`*

*✏️ (b) A Regressão Linear é pior classificadora porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 4 (descomente para ver) ───────────────────────────────
# print("(a) A Regressão Logística é praticamente sempre a melhor escolha aqui:")
# print("    ela foi desenhada para classificação, produz probabilidades de")
# print("    verdade (entre 0 e 1) e geralmente tem métricas melhores.")
# print()
# print("(b) A reta da Regressão Linear pode prever qualquer número (até negativo")
# print("    ou maior que 1), sem noção de probabilidade. A sigmoide da Regressão")
# print("    Logística sempre produz um valor entre 0 e 1, e se 'achata' perto dos")
# print("    extremos — exatamente o comportamento que uma classe binária precisa.")


**✏️ Minha reflexão sobre a aula:**

1. Diferença fundamental entre Regressão Linear e Regressão Logística: *...*

2. Por que a Regressão Linear não deveria ser usada para classificação: *...*

3. O que é o limiar de decisão e por que ele importa: *...*
